[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/16_embeddings_multimodales/16_proyecto_embeddings_multimodales.ipynb)

# Proyecto 16 · Políglota Visual: buscar en capturas y PDFs

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración estimada** | ≈ 2 h |
| **GPU recomendada** | No necesaria en `FAST_DEV_RUN=True` (CPU, ≈ 3–5 min). L4 recomendada para el reto con ColPali real |
| **Prerrequisitos** | Lección 16 (MaxSim, patches, OCR vs visual) y módulo 15 (métricas e IC) |
| **Conecta con** | Módulo 08 (patches) · 13 (MaxSim en ColBERT) · 15 (`poliglota_eval`) · 18 (servicio y coste) |

## 🏢 Contexto de negocio
Eres ML engineer en **Políglota Labs**. Soporte reporta que muchas respuestas están en **capturas de pantalla y PDFs** de la documentación, que el buscador de texto no «ve». La CTO te pide un prototipo de **Políglota Visual** y una recomendación: *¿OCR + texto, un modelo visual con late interaction, o ambos? ¿Y cuánto cuesta almacenarlo?*

## 📦 Dataset
Cada documento de Políglota (ES/EN) se convierte en una **captura sintética**: sus 8 términos clave (TF-IDF) impresos como palabras en una rejilla 3×4 (fuente real, desplazamientos aleatorios y **ruido de píxel**). Las queries y qrels son las de Políglota (`carga_corpus`; con datos reales, el vocabulario renderizable se limita a los 250 términos clave más frecuentes). Importante: las queries usan **sinónimos que no aparecen en la captura**, de modo que hace falta semántica.
En Colab con GPU puedes sustituir las capturas por páginas reales de **ViDoRe** y el codificador por **ColPali/ColQwen** (reto extra).

## 🎯 Entregables y criterios
| # | Entregable | Criterio objetivo |
|---|---|---|
| 1 | Pérdida de destilación visual→texto (TODO 1) | El codificador de patch alcanza ≥ 70 % de precisión de «lectura» de términos con σ=0 (medido: ≈ 80 % con 800 pasos) |
| 2 | `maxsim_paginas` correcto (TODO 2) | Pasa el test numérico (2.4) y las formas |
| 3 | `consulta_a_tokens` (TODO 3) | Pasa el test (tildes, OOV, vacía) |
| 4 | Evaluación con IC de 5 sistemas a σ=0 y σ=0.3 | nDCG@10 de cada sistema ≈ valor de referencia (± margen indicado) |
| 5 | Coste de almacenamiento (TODO 4) | Cifra correcta por 1 M de páginas para 4 esquemas |
| 6 | Recomendación para la CTO | Cita IC, ruido y coste |

## 0. Preparación (celdas dadas)

In [ ]:
!pip install -q numpy scipy scikit-learn matplotlib pandas torch pillow datasets

In [ ]:
import os, re, math, random, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED, FAST_DEV_RUN = 42, True
random.seed(SEED); np.random.seed(SEED)
import torch
torch.manual_seed(SEED)
if FAST_DEV_RUN: torch.set_num_threads(1)
device = "cuda" if torch.cuda.is_available() else "cpu"
AZUL, NARANJA, VERDE, GRIS, ROJO = "#1F6FEB", "#F28E2B", "#2CA02C", "#6B7280", "#D62728"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25})

Utilidades del corpus Políglota (`carga_corpus`: oficial → espejo → sintético), copia autocontenida del contrato común.

In [ ]:
# Corpus SINTÉTICO de Políglota (último recurso, mismo esquema que el real): 12 temas x 3 facetas.
# Por faceta: términos ES (5), EN (5), identificadores de código (3). Los 3 primeros términos son "centrales"
# (salen mucho en los documentos); los 2 últimos son sinónimos raros: las queries los usan -> hueco léxico.
TEMAS = {
 "cache": (["caché", "memoria", "datos"], ["cache", "memory", "data"], [
   ("ttl", "expiración|ttl|vencimiento|caducidad|refrescar", "expiration|ttl|timeout|lifetime|refresh", "set_ttl|expire_at|touch_key"),
   ("invalidacion", "invalidación|clave|purgar|obsoleto|desalojar", "invalidation|key|purge|stale|flush", "invalidate|purge_keys|delete_pattern"),
   ("evicion", "política|lru|desalojo|capacidad|recorte", "policy|lru|eviction|capacity|trim", "lru_evict|max_entries|pop_oldest")]),
 "basedatos": (["base", "datos", "consulta"], ["database", "data", "query"], [
   ("indices", "índice|b-tree|escaneo|selectividad|acelerar", "index|btree|scan|selectivity|speedup", "create_index|explain_plan|seq_scan"),
   ("transacciones", "transacción|commit|aislamiento|rollback|atomicidad", "transaction|commit|isolation|rollback|atomicity", "begin_tx|commit_tx|rollback_tx"),
   ("migraciones", "migración|esquema|columna|versionar|alterar", "migration|schema|column|versioning|alter", "run_migration|add_column|schema_version")]),
 "auth": (["usuario", "acceso", "seguridad"], ["user", "access", "security"], [
   ("tokens", "token|jwt|firma|sesión|testigo", "token|jwt|signature|session|bearer", "verify_jwt|sign_token|refresh_token"),
   ("oauth", "oauth|permiso|consentimiento|proveedor|delegar", "oauth|scope|consent|provider|delegate", "oauth_callback|request_scope|exchange_code"),
   ("hash", "contraseña|hash|sal|bcrypt|credencial", "password|hash|salt|bcrypt|credential", "hash_password|check_password|gen_salt")]),
 "contenedores": (["contenedor", "servicio", "despliegue"], ["container", "service", "deployment"], [
   ("imagenes", "imagen|capa|dockerfile|construir|empaquetar", "image|layer|dockerfile|build|package", "docker_build|pull_image|tag_image"),
   ("redes", "red|puerto|puente|exponer|conectar", "network|port|bridge|expose|connect", "create_network|publish_port|attach_bridge"),
   ("volumenes", "volumen|montaje|persistente|disco|compartir", "volume|mount|persistent|disk|share", "mount_volume|bind_path|prune_volumes")]),
 "testing": (["prueba", "código", "calidad"], ["test", "code", "quality"], [
   ("unitarias", "unitaria|aserción|fixture|función|verificar", "unit|assertion|fixture|function|verify", "assert_equal|run_unit|make_fixture"),
   ("mocks", "simulacro|mock|dependencia|sustituir|doble", "mock|stub|dependency|replace|double", "patch_call|make_mock|reset_mock"),
   ("cobertura", "cobertura|línea|rama|informe|medir", "coverage|line|branch|report|measure", "run_coverage|branch_report|min_cover")]),
 "concurrencia": (["hilo", "tarea", "ejecución"], ["thread", "task", "execution"], [
   ("hilos", "hilo|gil|paralelo|pool|lanzar", "thread|gil|parallel|pool|spawn", "start_thread|join_all|pool_map"),
   ("asincrono", "asíncrono|corrutina|await|bucle|esperar", "async|coroutine|await|loop|wait", "run_loop|gather_tasks|await_all"),
   ("bloqueos", "bloqueo|candado|interbloqueo|exclusión|sincronizar", "lock|mutex|deadlock|exclusion|synchronize", "acquire_lock|release_lock|try_lock")]),
 "api": (["api", "petición", "recurso"], ["api", "request", "resource"], [
   ("paginacion", "paginación|cursor|página|límite|desplazamiento", "pagination|cursor|page|limit|offset", "next_cursor|page_size|fetch_page"),
   ("versionado", "versión|compatibilidad|ruta|migrar|deprecar", "version|compatibility|route|migrate|deprecate", "api_version|route_v2|sunset_header"),
   ("limitacion", "cuota|tasa|límite|429|estrangular", "quota|rate|limit|429|throttle", "rate_limit|retry_after|token_bucket")]),
 "observabilidad": (["sistema", "registro", "evento"], ["system", "log", "event"], [
   ("logs", "log|nivel|formato|rotar|mensaje", "log|level|format|rotate|message", "get_logger|set_level|rotate_logs"),
   ("metricas", "métrica|contador|histograma|panel|alerta", "metric|counter|histogram|dashboard|alert", "inc_counter|observe_hist|export_metrics"),
   ("trazas", "traza|span|latencia|distribuido|correlación", "trace|span|latency|distributed|correlation", "start_span|inject_trace|trace_id")]),
 "git": (["repositorio", "cambio", "rama"], ["repository", "change", "branch"], [
   ("ramas", "rama|checkout|puntero|crear|cambiar", "branch|checkout|pointer|create|switch", "git_branch|git_checkout|list_branches"),
   ("merge", "fusión|merge|conflicto|resolver|integrar", "merge|conflict|resolve|integrate|combine", "git_merge|resolve_conflict|abort_merge"),
   ("rebase", "rebase|historial|reescribir|reaplicar|linealizar", "rebase|history|rewrite|replay|linearize", "git_rebase|squash_commits|pick_commit")]),
 "ml": (["modelo", "entrenamiento", "datos"], ["model", "training", "data"], [
   ("gradiente", "gradiente|descenso|pérdida|paso|optimizar", "gradient|descent|loss|step|optimize", "grad_step|zero_grad|clip_norm"),
   ("sobreajuste", "sobreajuste|regularización|generalizar|memorizar|dropout", "overfitting|regularization|generalize|memorize|dropout", "add_dropout|l2_penalty|early_stop"),
   ("validacion", "validación|pliegue|cruzada|partición|estimar", "validation|fold|cross|split|estimate", "kfold_split|val_score|holdout")]),
 "red": (["red", "paquete", "conexión"], ["network", "packet", "connection"], [
   ("tcp", "tcp|segmento|retransmisión|ventana|fiable", "tcp|segment|retransmission|window|reliable", "tcp_connect|set_window|resend_segment"),
   ("dns", "dns|dominio|resolución|registro|nombre", "dns|domain|resolution|record|name", "resolve_host|lookup_a|flush_dns"),
   ("tls", "tls|certificado|cifrado|apretón|confianza", "tls|certificate|encryption|handshake|trust", "tls_handshake|load_cert|verify_chain")]),
 "memoria": (["memoria", "proceso", "objeto"], ["memory", "process", "object"], [
   ("gc", "recolector|basura|referencia|ciclo|liberar", "collector|garbage|reference|cycle|free", "gc_collect|weak_ref|ref_count"),
   ("virtual", "virtual|página|dirección|paginación|fallo", "virtual|page|address|paging|fault", "map_page|page_fault|swap_out"),
   ("fugas", "fuga|crecimiento|perfilar|heap|retención", "leak|growth|profile|heap|retention", "heap_snapshot|track_alloc|find_leak")]),
}

In [ ]:
import os, random, re

PLANT_ES = ["Esta guía explica cómo {a} influye en {b} cuando se trabaja con {c}.",
            "Para resolver el problema de {a}, revisa primero {b} y luego {c}.",
            "En producción conviene vigilar {a}, porque {b} cambia el comportamiento de {c}.",
            "Un error habitual es mezclar {a} con {b} sin entender {c}.",
            "La documentación recomienda medir {a} antes de tocar {b} o {c}.",
            "Ejemplo práctico: ajustar {a} reduce el coste de {b} en {c}."]
PLANT_EN = ["This guide explains how {a} affects {b} when working with {c}.",
            "To solve the problem with {a}, first check {b} and then {c}.",
            "In production you should watch {a}, because {b} changes the behavior of {c}.",
            "A common mistake is mixing {a} with {b} without understanding {c}.",
            "The documentation recommends measuring {a} before touching {b} or {c}.",
            "Practical example: tuning {a} lowers the cost of {b} in {c}."]
QS_ES = ["cómo gestionar {s}", "problemas con {s}", "guía de {s}", "qué hacer con {s}"]
QS_EN = ["how to handle {s}", "problems with {s}", "guide to {s}", "what to do about {s}"]

def _slot(rng, core, rare, shared, other):
    r = rng.random()
    return rng.choice(core) if r < .62 else rng.choice(shared) if r < .80 else rng.choice(rare) if r < .86 else rng.choice(other)

def genera_sintetico(n_temas: int = 8, docs_por_faceta: int = 4, seed: int = 42) -> dict:
    """Mismo esquema que el corpus real. Relevancia: docs de la misma faceta e idioma (code: faceta, queries 'code')."""
    rng = random.Random(seed)
    nombres = list(TEMAS)[:n_temas]
    docs, queries, qrels = [], [], {}
    for t in nombres:
        es_sh, en_sh, facetas = TEMAS[t]
        for fid, es, en, code in facetas:
            otros = [x for n in nombres if n != t for x in TEMAS[n][2][0][1].split("|")[:3]]
            otros_en = [x for n in nombres if n != t for x in TEMAS[n][2][0][2].split("|")[:3]]
            es, en, code = es.split("|"), en.split("|"), code.split("|")
            rel = {"es": {}, "en": {}, "code": {}}
            for lang, terms, shared, oth, plant in (("es", es, es_sh, otros, PLANT_ES), ("en", en, en_sh, otros_en, PLANT_EN)):
                for i in range(docs_por_faceta):
                    sents = [rng.choice(plant).format(a=_slot(rng, terms[:3], terms[3:], shared, oth),
                                                      b=_slot(rng, terms[:3], terms[3:], shared, oth),
                                                      c=_slot(rng, terms[:3], terms[3:], shared, oth)) for _ in range(3)]
                    did = f"{lang}-{t}.{fid}-{i}"
                    docs.append({"id": did, "lang": lang, "text": " ".join(sents)}); rel[lang][did] = 1
            for i in range(max(2, docs_por_faceta - 1)):
                fn = rng.choice(code); aux = rng.choice(code); w = rng.choice(en[:3])
                did = f"code-{t}.{fid}-{i}"
                docs.append({"id": did, "lang": "code", "text": f"def {fn}(x, n={rng.randint(2, 9)}):\n    \"\"\"{w} helper for {rng.choice(en_sh)}\"\"\"\n    return {aux}(x, {rng.choice(code)})"})
                rel["code"][did] = 1
            for lang, terms, qs in (("es", es, QS_ES), ("en", en, QS_EN), ("code", en, QS_EN)):
                s = " ".join(([rng.choice(terms[:3])] if rng.random() < .5 else []) + rng.sample(terms[3:], 2))   # 50%: 1 central + 2 raros
                qid = f"q-{lang}-{t}.{fid}"
                queries.append({"qid": qid, "lang": lang, "text": rng.choice(qs).format(s=s) + (" function" if lang == "code" else "")})
                qrels[qid] = dict(rel[lang])
    rng.shuffle(docs)
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": "synthetic"}

In [ ]:
def _carga_real(n_docs: int, seed: int = 42):
    """Cadena OFICIAL: HF `wikimedia/wikipedia` (ES/EN, streaming) + CodeSearchNet Python. Lanza excepción si no hay red/HF.
    Espejo: define HF_ENDPOINT=https://hf-mirror.com ANTES de importar `datasets` (reinicia el runtime) y repite."""
    from datasets import load_dataset
    docs, queries, qrels = [], [], {}
    for lang, cfg in (("es", "20231101.es"), ("en", "20231101.en")):
        ds = load_dataset("wikimedia/wikipedia", cfg, split="train", streaming=True).shuffle(seed=seed, buffer_size=2000)
        got = 0
        for row in ds:
            body = re.sub(r"\s+", " ", row["text"].split("\n\n")[0]).strip()
            if len(body) < 200:
                continue
            did, qid = f"{lang}-{row['id']}", f"q-{lang}-{row['id']}"
            docs.append({"id": did, "lang": lang, "text": body[:700]})
            queries.append({"qid": qid, "lang": lang, "text": row["title"]}); qrels[qid] = {did: 1}
            got += 1
            if got >= n_docs: break
    cs = load_dataset("code-search-net/code_search_net", "python", split="test", streaming=True)
    got = 0
    for i, row in enumerate(cs):
        doc_str = (row.get("func_documentation_string") or "").strip().split("\n")[0]
        code = row.get("func_code_string") or row.get("whole_func_string") or ""
        if len(doc_str) < 25 or not code: continue
        did, qid = f"code-{i}", f"q-code-{i}"
        docs.append({"id": did, "lang": "code", "text": code[:700]})
        queries.append({"qid": qid, "lang": "code", "text": doc_str}); qrels[qid] = {did: 1}
        got += 1
        if got >= n_docs: break
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": "real"}

def carga_corpus(fast_dev: bool = True) -> dict:
    """Contrato Políglota: {"docs":[{id,lang,text}], "queries":[{qid,lang,text}], "qrels":{qid:{doc_id:1}}, "source":"real"|"synthetic"}.
    Cadena: oficial (HF) -> espejo (HF_ENDPOINT) -> sintético. NOTA: la versión real usa queries = título del artículo /
    docstring (golden set auto-supervisado, más fácil que uno humano); el sintético usa facetas con huecos léxicos."""
    if os.environ.get("POLIGLOTA_FORCE_SYNTHETIC") != "1":
        try:
            return _carga_real(n_docs=150 if fast_dev else 1500)
        except Exception as e:  # sin red, proxy, dataset movido...
            print(f"⚠️ Datos reales no disponibles ({type(e).__name__}: {str(e)[:80]}) -> corpus SINTÉTICO.")
    return genera_sintetico(n_temas=8 if fast_dev else 12)

Métricas y estadística del módulo 15 (`poliglota_eval.py`), retrievers de CPU, y la librería de páginas/patches/MaxSim de la lección 16.

In [ ]:
import math
import numpy as np

def _dedup(ranked_ids):
    """Conserva la primera aparición: un doc repetido NO puntúa dos veces."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d); out.append(d)
    return out

def _gain(r: float, exp_gain: bool) -> float:
    return (2.0 ** r - 1.0) if exp_gain else float(r)

def dcg_at_k(ranked_ids, rels: dict, k: int = 10, exp_gain: bool = False) -> float:
    """DCG@k = sum_i gain(rel_i) / log2(i+1), i = 1..k. rels: doc_id -> relevancia (>0 relevante; ausente = 0)."""
    return sum(_gain(rels.get(d, 0), exp_gain) / math.log2(i + 2)
               for i, d in enumerate(_dedup(ranked_ids)[:k]) if rels.get(d, 0) > 0)

def ndcg_at_k(ranked_ids, rels: dict, k: int = 10, exp_gain: bool = False) -> float:
    """nDCG@k = DCG@k / IDCG@k. IDCG usa los mejores k de TODOS los juzgados. Sin relevantes -> 0.0 (convención)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(_gain(r, exp_gain) / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k, exp_gain) / idcg

def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """|relevantes en top-k| / |relevantes|. Sin relevantes -> 0.0."""
    relevant = {d for d, r in rels.items() if r > 0}
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant) if relevant else 0.0

def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k; 0 si no hay ninguno."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0

def precision_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Divide SIEMPRE por k (como trec_eval), aunque la lista sea más corta."""
    return sum(rels.get(d, 0) > 0 for d in _dedup(ranked_ids)[:k]) / k

def rank_from_scores(scores, doc_ids, k=None):
    """Orden por score desc; empates -> orden del corpus (determinista). Sin esto, 'argsort' decide por ti."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    return [doc_ids[i] for i in (order[:k] if k else order)]

METRICS = {"ndcg": ndcg_at_k, "recall": recall_at_k, "mrr": mrr_at_k, "precision": precision_at_k}

def evaluate_per_query(run: dict, qrels: dict, k: int = 10, metrics=("ndcg", "recall", "mrr")) -> dict:
    """run: {qid: [doc_id ordenados]}. Devuelve {"qids": [...], metric: np.ndarray}. Excluye queries sin relevantes
    (convención trec_eval); una query sin ranking en `run` puntúa 0 (NO se descarta: sería hacer trampa)."""
    qids = [q for q, rels in qrels.items() if any(r > 0 for r in rels.values())]
    out = {"qids": qids}
    for m in metrics:
        out[m] = np.array([METRICS[m](run.get(q, []), qrels[q], k) for q in qids])
    return out

def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    per = evaluate_per_query(run, qrels, k)
    return {f"{m}@{k}": float(per[m].mean()) for m in ("ndcg", "recall", "mrr")} | {"n_queries": len(per["qids"])}

In [ ]:
def bootstrap_ci(values, n_boot: int = 2000, alpha: float = 0.05, seed: int = 42):
    """IC percentil de la media remuestreando QUERIES (la unidad de muestreo es la query, no el par query-doc)."""
    v = np.asarray(values, dtype=float)
    idx = np.random.default_rng(seed).integers(0, len(v), size=(n_boot, len(v)))
    means = v[idx].mean(axis=1)
    return float(v.mean()), float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))

def paired_bootstrap(a, b, n_boot: int = 10000, alpha: float = 0.05, seed: int = 42) -> dict:
    """Test pareado A vs B sobre las MISMAS queries. IC percentil de la diferencia media y p-valor bilateral
    por 'shift' (se centra la diferencia en 0 bajo H0 y se cuenta cuántas veces el remuestreo supera lo observado)."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    means = d[idx].mean(axis=1)
    null = (d - d.mean())[idx].mean(axis=1)
    p = (np.sum(np.abs(null) >= abs(d.mean())) + 1) / (n_boot + 1)
    return {"diff": float(d.mean()), "ci_low": float(np.quantile(means, alpha / 2)),
            "ci_high": float(np.quantile(means, 1 - alpha / 2)), "p_value": float(p)}

def paired_permutation(a, b, n_perm: int = 10000, seed: int = 42) -> float:
    """Test de aleatorización pareado (sign-flip): bajo H0 el signo de cada diferencia por query es intercambiable."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    signs = np.random.default_rng(seed).choice([-1.0, 1.0], size=(n_perm, len(d)))
    stat = np.abs((signs * d).mean(axis=1))
    return float((np.sum(stat >= abs(d.mean()) - 1e-12) + 1) / (n_perm + 1))

def holm(pvalues: dict, alpha: float = 0.05) -> dict:
    """Corrección de Holm-Bonferroni para comparaciones múltiples. Devuelve {nombre: (p_ajustado, rechaza_H0)}."""
    items = sorted(pvalues.items(), key=lambda kv: kv[1])
    m, out, running = len(items), {}, 0.0
    for i, (name, p) in enumerate(items):
        running = max(running, min(1.0, (m - i) * p))
        out[name] = (running, running < alpha)
    return out

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

def tok_simple(text: str) -> list[str]:
    """Token léxico: minúsculas + regex de palabras (snake_case queda como UN solo token; lo veremos fallar en código)."""
    return re.findall(r"\w+", text.lower())

def tok_code(text: str) -> list[str]:
    """Igual, pero parte snake_case y camelCase (set_ttl -> set, ttl). Es tokenización con criterio de dominio."""
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text).replace("_", " ")
    return re.findall(r"[^\W_]+", text.lower())

class BM25:
    """BM25 (Robertson/Lucene idf) con matriz dispersa. k1=1.2, b=0.75."""
    def __init__(self, tokenizer=tok_simple, k1=1.2, b=0.75):
        self.tok, self.k1, self.b = tokenizer, k1, b
    def fit(self, texts, ids):
        self.ids = list(ids)
        self.cv = CountVectorizer(tokenizer=self.tok, lowercase=False, token_pattern=None)
        X = self.cv.fit_transform(texts).tocsr().astype(float)
        self.N = X.shape[0]; self.dl = np.asarray(X.sum(1)).ravel(); self.avg = self.dl.mean()
        df = np.asarray((X > 0).sum(0)).ravel()
        self.idf = np.log(1 + (self.N - df + 0.5) / (df + 0.5))
        self.X = X.tocsc()
        return self
    def scores(self, query: str) -> np.ndarray:
        s = np.zeros(self.N)
        for j in set(self.cv.transform([query]).indices):
            col = self.X[:, j].toarray().ravel()
            s += self.idf[j] * col * (self.k1 + 1) / (col + self.k1 * (1 - self.b + self.b * self.dl / self.avg))
        return s

class LSI:
    """Retriever 'denso' sin descargas: TF-IDF sublineal + SVD truncada (LSA) + coseno. Hace de embedder de CPU."""
    def __init__(self, dim=48, seed=42):
        self.dim, self.seed = dim, seed
    def fit(self, texts, ids):
        self.ids = list(ids)
        self.tf = TfidfVectorizer(sublinear_tf=True)
        T = self.tf.fit_transform(texts)
        self.svd = TruncatedSVD(min(self.dim, T.shape[1] - 1), random_state=self.seed)
        self.Z = self._norm(self.svd.fit_transform(T))
        return self
    @staticmethod
    def _norm(Z): return Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)
    def scores(self, query: str) -> np.ndarray:
        return self.Z @ self._norm(self.svd.transform(self.tf.transform([query])))[0]

def rrf_scores(score_lists, k: int = 60) -> np.ndarray:
    """Reciprocal Rank Fusion: sum 1/(k + rango). Ignora la escala de cada score (ventaja y límite)."""
    out = np.zeros(len(score_lists[0]))
    for s in score_lists:
        order = np.lexsort((np.arange(len(s)), -np.asarray(s)))
        out[order] += 1.0 / (k + np.arange(len(s)) + 1)
    return out

def build_run(scorer, queries, doc_ids, topn: int = 100) -> dict:
    """scorer(texto_query) -> vector de scores alineado con doc_ids. Devuelve run {qid: [doc_ids top-n]}."""
    return {q["qid"]: rank_from_scores(scorer(q["text"]), doc_ids, topn) for q in queries}

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

FORMAS = ["circle", "square", "triangle", "cross"]; COLORES = ["red", "green", "blue", "yellow"]
RGB = {"red": (.9, .1, .1), "green": (.1, .7, .1), "blue": (.1, .2, .9), "yellow": (.95, .85, .1)}
ES_F = {"circle": ("un", "círculo"), "square": ("un", "cuadrado"), "triangle": ("un", "triángulo"), "cross": ("una", "cruz")}
ES_C = {"red": ("rojo", "roja"), "green": ("verde", "verde"), "blue": ("azul", "azul"), "yellow": ("amarillo", "amarilla")}
CONCEPTOS = [(f, c) for f in FORMAS for c in COLORES]          # 16 conceptos

def dibuja(forma: str, color: str, rng, S: int = 32, centro=None) -> np.ndarray:
    """Imagen SxSx3 float32 con una figura de color sobre fondo gris con ruido. Posición y tamaño aleatorios."""
    y, x = np.mgrid[0:S, 0:S]; cx, cy = rng.integers(int(S*.34), int(S*.66), 2) if centro is None else centro; r = rng.integers(max(2, int(S*.19)), int(S*.31))
    if forma == "circle": m = (x - cx)**2 + (y - cy)**2 < r * r
    elif forma == "square": m = (abs(x - cx) < r * .8) & (abs(y - cy) < r * .8)
    elif forma == "triangle": m = (y >= cy - r) & (y <= cy + r) & (abs(x - cx) <= (y - (cy - r)) / 2 + .5)
    else: m = ((abs(x - cx) < max(1, r / 3)) & (abs(y - cy) < r)) | ((abs(y - cy) < max(1, r / 3)) & (abs(x - cx) < r))
    img = np.full((S, S, 3), .5) + rng.normal(0, .04, (S, S, 3)); img[m] = np.array(RGB[color]) + rng.normal(0, .04, (m.sum(), 3))
    return np.clip(img, 0, 1).astype(np.float32)

def caption(forma: str, color: str, lang: str, rng) -> list:
    """Caption como lista de palabras, en ES o EN, con 3 plantillas por idioma (concordancia de género en ES)."""
    if lang == "en":
        return rng.choice(["a {c} {s}", "picture of a {c} {s}", "a photo of a {c} {s}"]).format(c=color, s=forma).split()
    art, s = ES_F[forma]; col = ES_C[color][1 if art == "una" else 0]
    return rng.choice(["{a} {s} {c}", "imagen de {a} {s} {c}", "una foto de {a} {s} {c}"]).format(a=art, s=s, c=col).split()

def make_dataset(n_per: int, seed: int):
    """Devuelve (imágenes, captions, etiqueta de concepto, idioma). Cada imagen lleva UNA caption (ES o EN al azar)."""
    rng = np.random.default_rng(seed); X, T, Y, L = [], [], [], []
    for k, (f, c) in enumerate(CONCEPTOS):
        for _ in range(n_per):
            lang = "es" if rng.random() < .5 else "en"
            X.append(dibuja(f, c, rng)); T.append(caption(f, c, lang, rng)); Y.append(k); L.append(lang)
    return np.stack(X), T, np.array(Y), L

In [ ]:
import math, unicodedata
from PIL import Image, ImageDraw, ImageFont

try: FUENTE = ImageFont.load_default(size=11)      # Pillow >= 10.1 trae una fuente escalable
except TypeError: FUENTE = ImageFont.load_default()
CH, CW, FILAS, COLS = 16, 96, 3, 4                  # celda de 16x96 px; página = 3x4 celdas = 12 "patches"

def plano(s: str) -> str:
    """La fuente por defecto no tiene tildes: las plegamos (la limitación es de la FUENTE, no del enfoque)."""
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")

def celda_palabra(w: str, rng=None) -> np.ndarray:
    img = Image.new("L", (CW, CH), 255); dx = int(rng.integers(0, 6)) if rng is not None else 0; dy = int(rng.integers(0, 3)) if rng is not None else 0
    ImageDraw.Draw(img).text((2 + dx, 1 + dy), w, font=FUENTE, fill=0); return np.asarray(img, dtype=np.float32) / 255.

def celda_icono(k: int, rng) -> np.ndarray:
    """Icono (forma de color) 16x16 sobre una celda blanca: contenido NO textual (como un logo o una figura de la página)."""
    f, c = CONCEPTOS[k]; ic = dibuja(f, c, rng, S=16); lum = {"red": .35, "green": .6, "blue": .1, "yellow": .85}[c]   # el color -> nivel de gris
    mask = np.abs(ic - .5).max(-1) > .25; cel = np.ones((CH, CW), np.float32); x0 = int(rng.integers(0, CW - 16))
    cel[:, x0:x0 + 16] = np.where(mask, lum, 1.0); return cel

def celda_vacia() -> np.ndarray: return np.ones((CH, CW), np.float32)

def pagina_a_imagen(celdas, ruido: float, rng) -> np.ndarray:
    """Une 12 celdas en una imagen de 48x384 px y añade ruido gaussiano de píxel (escaneo/compresión)."""
    a = np.stack(celdas).reshape(FILAS, COLS, CH, CW).transpose(0, 2, 1, 3).reshape(FILAS * CH, COLS * CW)
    return a + rng.normal(0, ruido, a.shape).astype(np.float32)

def parchea(pagina: np.ndarray) -> np.ndarray:
    """Imagen de página -> (12, 16, 96): los 'patches' (celdas). Un ViT real usa cuadrados de 14-16 px; aquí cada patch es una celda."""
    return pagina.reshape(FILAS, CH, COLS, CW).transpose(0, 2, 1, 3).reshape(FILAS * COLS, CH, CW)

class RedPatch(nn.Module):
    """Codificador de patch: CNN -> vector L2-normalizado de dimensión D (como la proyección a 128-d de ColPali, en pequeño)."""
    def __init__(self, D: int = 32):
        super().__init__()
        self.c = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                               nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((2, 6)), nn.Flatten(), nn.Linear(64 * 12, D))
    def forward(self, x): return F.normalize(self.c(x.unsqueeze(1)), dim=-1)

def _maxsim_dado(Q: np.ndarray, P: np.ndarray) -> np.ndarray:
    """Late interaction (ColBERT/ColPali): score(q, pág) = sum_i max_j <q_i, p_j>.
    Q: (nq, D) tokens de la query; P: (n_págs, n_patches, D). Devuelve (n_págs,). Vectorizado con einsum."""
    return np.einsum("qd,npd->nqp", Q, P).max(axis=2).sum(axis=1)

def ocr_plantillas(palabras, rng_seed=0):
    """OCR 'clásico' por correlación: plantillas de cada palabra en TODOS los desplazamientos (dx, dy). Es un filtro adaptado,
    casi óptimo contra ruido gaussiano para texto renderizado con esta fuente. Devuelve (T, etiqueta de cada plantilla)."""
    T, lab = [celda_vacia().ravel()], [0]
    for k, w in enumerate(palabras, 1):
        for dx in range(6):
            for dy in range(3):
                img = Image.new("L", (CW, CH), 255); ImageDraw.Draw(img).text((2 + dx, 1 + dy), w, font=FUENTE, fill=0)
                T.append(np.asarray(img, dtype=np.float32).ravel() / 255.); lab.append(k)
    return np.stack(T), np.array(lab)

def ocr_pagina(parches: np.ndarray, T: np.ndarray, lab: np.ndarray) -> np.ndarray:
    """Reconoce cada patch como la palabra de la plantilla más cercana (L2). Los iconos se 'leen' mal: no son texto."""
    d = torch.cdist(torch.tensor(parches.reshape(len(parches), -1)), torch.tensor(T)); return lab[d.argmin(1).numpy()]

In [ ]:
corpus = carga_corpus(fast_dev=FAST_DEV_RUN)
docs = [d for d in corpus["docs"] if d["lang"] != "code"]            # las "capturas" son de documentación en ES/EN
ids, txt = [d["id"] for d in docs], [d["text"] for d in docs]; N = len(docs)
queries = [q for q in corpus["queries"] if q["lang"] != "code"]; qrels = {q["qid"]: corpus["qrels"][q["qid"]] for q in queries}
print(corpus["source"], N, "documentos-página ·", len(queries), "queries")

# "Profesor" de texto: LSI-48 (embedder de texto). Sus vectores de TÉRMINO serán el espacio de los tokens de la query.
lsi = LSI(48, SEED).fit(txt, ids); bm25_full = BM25(tok_simple).fit(txt, ids)
comp = lsi.svd.components_.T; comp = comp / (np.linalg.norm(comp, axis=1, keepdims=True) + 1e-9)
vec_termino = {}
for t, j in lsi.tf.vocabulary_.items(): vec_termino.setdefault(plano(t), comp[j])      # término (sin tildes) -> vector 48-d

# Términos clave de cada documento (top TF-IDF) -> lo que se "imprime" en su captura
Xtf = lsi.tf.transform(txt); inv = {j: t for t, j in lsi.tf.vocabulary_.items()}; dfreq = np.asarray((Xtf > 0).sum(0)).ravel()
def terminos_clave(i, k=8):
    fila = Xtf[i].toarray().ravel(); out = []
    for j in np.argsort(-fila):
        t = plano(inv[j])
        if fila[j] == 0: break
        if re.fullmatch(r"[a-z]{3,12}", t) and 3 <= dfreq[j] <= 0.2 * N and t not in out: out.append(t)
        if len(out) >= k: break
    return out
CLAVE = [terminos_clave(i) for i in range(N)]
VOC = sorted({t for k in CLAVE for t in k})[:250]; w2c = {w: i + 1 for i, w in enumerate(VOC)}      # clase 0 = celda vacía
CLAVE = [[w for w in k if w in w2c] for k in CLAVE]
print(f"vocabulario renderizable: {len(VOC)} términos · {np.mean([len(k) for k in CLAVE]):.1f} términos clave por página")

In [ ]:
rr = np.random.default_rng(7)
def render_doc(i, ruido=0.0):
    """Captura sintética del documento i: sus términos clave impresos en celdas aleatorias de una rejilla 3x4."""
    g = np.zeros(12, int); g[rr.choice(12, len(CLAVE[i]), replace=False)] = [w2c[w] for w in CLAVE[i]]
    return pagina_a_imagen([celda_vacia() if c == 0 else celda_palabra(VOC[c - 1], rr) for c in g], ruido, rr)

def parches_corpus(ruido: float) -> np.ndarray:
    return np.stack([parchea(render_doc(i, ruido)) for i in range(N)])          # (N, 12, 16, 96)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 2.5))
for a, (i, rz) in zip(ax, ((0, 0.0), (0, 0.3), (60, 0.0))):
    a.imshow(render_doc(i, rz), cmap="gray", vmin=0, vmax=1); a.set_title(f"captura del doc {ids[i]} · σ={rz}", fontsize=9); a.axis("off")
plt.tight_layout(); plt.show(); print("términos clave del doc 0:", CLAVE[0])

> 👀 **Qué debes observar:** la misma captura sin ruido y con σ = 0.3, y la de otro documento. Antes de seguir, **apuesta**: ¿quién sufrirá más con el ruido, el OCR por plantillas (que solo tiene que acertar palabras de un vocabulario cerrado) o el retriever visual (que compara parches de píxeles)? La sección 3 lo mide; si el resultado contradice tu intuición, busca la causa en cómo está construido cada sistema, no en el azar.

## 1. Destilación visual → texto (TODO 1)
El «profesor» de texto (LSI) da a cada término un vector 48-d. El **estudiante visual** (`RedPatch`) debe mapear la imagen de una palabra a *ese* vector, de modo que un token de query y un patch de la página queden en el mismo espacio (en ColPali lo consigue el VLM entrenado contrastivamente).
Implementa `perdida_destilacion`: logits = similitud coseno entre el embedding del patch y las **filas fijas** de la tabla de objetivos (la fila 0 es el embedding de la celda vacía), dividida por una temperatura `tau` (0.05), y entropía cruzada con la clase correcta.

<details><summary>🪜 Pista 1 (la idea)</summary>

Es una clasificación sobre el vocabulario, pero con los *pesos* fijados a los vectores del profesor (normalizados) en lugar de aprendidos. `tau` pequeña afila el softmax.
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

`logits = F.normalize(z, dim=-1) @ F.normalize(objetivos, dim=-1).T / tau`; `return F.cross_entropy(logits, y)`. Aquí `z` ya viene normalizado de `RedPatch`.
</details>

In [ ]:
def perdida_destilacion(z: torch.Tensor, y: torch.Tensor, objetivos: torch.Tensor, tau: float = 0.05) -> torch.Tensor:
    """z: (B, D) embeddings de patch; y: (B,) clase (0 = vacía, k = término k); objetivos: (V+1, D) vectores del profesor."""
    # TODO 1: entropía cruzada sobre logits coseno / tau
    raise NotImplementedError("TODO 1: perdida_destilacion")

def test_perdida():
    z = F.normalize(torch.eye(3)[:, :3] + 0.01, dim=-1); obj = torch.eye(3); y = torch.tensor([0, 1, 2])
    l_ok = perdida_destilacion(z, y, obj); l_mal = perdida_destilacion(z, torch.tensor([1, 2, 0]), obj)
    assert l_ok.ndim == 0 and l_ok < l_mal, "la clase correcta debe tener menos pérdida que una incorrecta"
    z.requires_grad_(True); perdida_destilacion(z, y, obj).backward(); assert z.grad is not None
    return "✅ test_perdida OK"
def corre(f):
    try: print(f())
    except NotImplementedError as e: print(f"⏳ pendiente: {e}")
    except AssertionError as e: print(f"❌ falló: {e}")
    except Exception as e: print(f"❌ {type(e).__name__}: {e}")
corre(test_perdida)

In [ ]:
D = 48
OBJ = torch.tensor(np.stack([np.zeros(D)] + [vec_termino.get(w, np.zeros(D)) for w in VOC]).astype(np.float32))     # fila 0 la aprende la red

def muestra_celdas(n, rng, ruido_max=.25):
    ys = rng.integers(0, len(VOC) + 1, n); xs = []
    for y in ys:
        c = celda_vacia() if y == 0 else celda_palabra(VOC[y - 1], rng); xs.append(c + rng.normal(0, float(rng.uniform(0, ruido_max)), c.shape).astype(np.float32))
    return torch.tensor(np.stack(xs)), torch.tensor(ys)

def entrena_estudiante(pasos=800, seed=0):
    """(Dado) Bucle de entrenamiento: usa TU perdida_destilacion."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed); red = RedPatch(D); vacia = nn.Parameter(torch.randn(1, D) * .1)
    opt = torch.optim.Adam(list(red.parameters()) + [vacia], lr=3e-3); hist = []
    for _ in range(pasos):
        x, y = muestra_celdas(128, rng); obj = torch.cat([vacia, OBJ[1:]])
        loss = perdida_destilacion(red(x), y, obj); opt.zero_grad(); loss.backward(); opt.step(); hist.append(loss.item())
    return red.eval(), hist
try:
    t0 = time.time(); RED, hist = entrena_estudiante(800 if FAST_DEV_RUN else 2500)
    x, y = muestra_celdas(600, np.random.default_rng(99), ruido_max=0.0)
    with torch.no_grad(): z = RED(x); acc_lectura = float(((F.normalize(z, dim=-1) @ F.normalize(OBJ, dim=-1).T)[:, 1:].argmax(1) + 1 == y)[y > 0].float().mean())
    print(f"entrenado en {time.time()-t0:.0f}s · pérdida final {np.mean(hist[-20:]):.3f} · precisión de lectura de palabras (σ=0): {acc_lectura:.3f}")
except NotImplementedError as e:
    RED, acc_lectura = None, None; print("⏳ pendiente:", e)

## 2. MaxSim y tokens de query (TODOs 2 y 3)
$$\mathrm{score}(q,\text{página})=\sum_i\max_j\langle q_i,p_j\rangle$$
Implementa **`maxsim_paginas(Q, P)`** vectorizado (`Q`: `(nq, D)`; `P`: `(n_págs, n_patches, D)` → `(n_págs,)`) y **`consulta_a_tokens`**: minúsculas, `\w+`, sin tildes (`plano`), vector del profesor para cada término en `vec_termino` y descarte de los desconocidos (devuelve `None` si no queda ninguno).

<details><summary>🪜 Pista 1 (la idea)</summary>

Con `einsum` obtienes la matriz de similitudes `(n_págs, nq, n_patches)`; el máximo va sobre el último eje y la suma sobre tokens.
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

`np.einsum('qd,npd->nqp', Q, P).max(axis=2).sum(axis=1)`. Para los tokens: `[plano(w) for w in re.findall(r'\w+', q.lower())]` y `np.stack([vec_termino[w] for w in ws if w in vec_termino])`.
</details>

In [ ]:
def maxsim_paginas(Q: np.ndarray, P: np.ndarray) -> np.ndarray:
    # TODO 2
    raise NotImplementedError("TODO 2: maxsim_paginas")

def consulta_a_tokens(q: str):
    # TODO 3
    raise NotImplementedError("TODO 3: consulta_a_tokens")

def test_maxsim_tokens():
    S = np.array([[.9, .1, .0, .2], [.1, .8, .3, .0], [.0, .2, .7, .1]])      # 3 tokens x 4 patches (ejemplo de la lección)
    Qv = np.eye(4)[:3]; P4 = np.zeros((2, 4, 4)); P4[0, :, :3] = S.T          # tokens = e1..e3; página 0 con esas similitudes, página 1 vacía
    s = maxsim_paginas(Qv, P4); assert s.shape == (2,) and abs(s[0] - 2.4) < 1e-9 and abs(s[1]) < 1e-12, "MaxSim del ejemplo = .9+.8+.7"
    t = consulta_a_tokens("Cómo configurar la CACHÉ de índice"); assert t is None or (t.ndim == 2 and t.shape[1] == 48)
    assert consulta_a_tokens("zzzz qqqq") is None, "palabras desconocidas -> None"
    a, b = consulta_a_tokens("caché"), consulta_a_tokens("cache"); assert a is not None and np.allclose(a, b), "las tildes se pliegan"
    return "✅ test_maxsim_tokens OK"
corre(test_maxsim_tokens)

## 3. Cinco sistemas, dos niveles de ruido (celdas dadas + tu evaluación)
| Sistema | Qué hace |
|---|---|
| BM25 sobre el **texto completo** | referencia: lo que hay si tuvieras el texto original (no hay OCR) |
| **OCR + BM25** | reconoce las palabras de la captura (OCR clásico por plantillas) y hace BM25: puramente léxico |
| **OCR + LSI** | reconoce las palabras y las embebe con el LSI del corpus |
| **MaxSim visual** | tu estudiante visual + tu MaxSim sobre los patches |
| **Vector medio visual** | un solo vector por página (media de patches) |

In [ ]:
def construye_sistemas(ruido):
    P = parches_corpus(ruido)
    with torch.no_grad(): PE = RED(torch.tensor(P.reshape(-1, CH, CW))).reshape(N, 12, -1).numpy()
    TPL, LAB = ocr_plantillas(VOC); ocr = np.stack([ocr_pagina(P[i], TPL, LAB) for i in range(N)])
    ocr_txt = [" ".join(VOC[c - 1] for c in o if c > 0) for o in ocr]; bm_ocr = BM25(tok_simple).fit(ocr_txt, ids)
    Zo = lsi.svd.transform(lsi.tf.transform(ocr_txt)); Zo /= np.linalg.norm(Zo, axis=1, keepdims=True) + 1e-9
    def ocr_lsi(q):
        z = lsi.svd.transform(lsi.tf.transform([q]))[0]; return Zo @ (z / (np.linalg.norm(z) + 1e-9))
    def maxsim_vis(q):
        Q = consulta_a_tokens(q); return maxsim_paginas(Q, PE) if Q is not None else np.zeros(N)
    def media_vis(q):
        Q = consulta_a_tokens(q); return PE.mean(1) @ Q.mean(0) if Q is not None else np.zeros(N)
    return {"BM25 texto completo": bm25_full.scores, "OCR + BM25": bm_ocr.scores, "OCR + LSI": ocr_lsi, "MaxSim visual": maxsim_vis, "Vector medio visual": media_vis}

def evalua_sistemas(ruido):
    sis = construye_sistemas(ruido); runs = {n: build_run(f, queries, ids, 100) for n, f in sis.items()}
    return {n: evaluate_per_query(r, qrels, 10) for n, r in runs.items()}

### TODO 4 · Informe con IC y coste de almacenamiento
Para cada nivel de ruido calcula **nDCG@10 con IC95 %** (`bootstrap_ci`) de los cinco sistemas, y un **test pareado** de *MaxSim visual* contra *OCR + BM25* (`paired_bootstrap`). Después implementa `gb_por_millon(n_vec, dim, bits)` y completa la tabla de coste para 1 M de páginas.

<details><summary>🪜 Pista 1 (la idea)</summary>

`PER = evalua_sistemas(0.0)` te da `PER[sistema]['ndcg']` (un valor por query). Recorre los sistemas con `bootstrap_ci`. Coste: bytes = n_vec · dim · bits / 8.
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

`gb_por_millon = n_vec * dim * bits / 8 * 1e6 / 1e9` (GB decimales). ColPali: 1030 × 128 × 16 bits; un vector denso 1024-d fp16: 1 × 1024 × 16; binario: 1 × 1024 × 1.
</details>

In [ ]:
def gb_por_millon(n_vec: int, dim: int, bits: int) -> float:
    """GB (10^9 bytes) para un millón de páginas."""
    # TODO 4a
    raise NotImplementedError("TODO 4a: gb_por_millon")

def informe(ruido):
    """Devuelve {'ic': {sistema: (media, lo, hi)}, 'test': dict de paired_bootstrap MaxSim vs OCR+BM25, 'per': per-query}."""
    # TODO 4b: usa evalua_sistemas(ruido), bootstrap_ci y paired_bootstrap
    raise NotImplementedError("TODO 4b: informe")

def test_coste():
    assert abs(gb_por_millon(1030, 128, 16) - 263.68) < 1e-6 and abs(gb_por_millon(1, 1024, 16) - 2.048) < 1e-9 and abs(gb_por_millon(1, 1024, 1) - 0.128) < 1e-9
    return "✅ test_coste OK"
corre(test_coste)
RES = {}
try:
    for rz in (0.0, 0.3):
        RES[rz] = informe(rz)
        print(f"\n=== ruido σ={rz} ==="); [print(f"  {n:22s} nDCG@10 = {m:.3f}  IC95 [{lo:.3f}, {hi:.3f}]") for n, (m, lo, hi) in RES[rz]["ic"].items()]
        t = RES[rz]["test"]; print(f"  MaxSim − OCR+BM25: Δ={t['diff']:+.3f} IC95[{t['ci_low']:+.3f},{t['ci_high']:+.3f}] p={t['p_value']:.4f}")
    ESQ = {"ColPali-like (1030×128 fp16)": gb_por_millon(1030, 128, 16), "Nuestro juguete (12×48 fp32)": gb_por_millon(12, 48, 32), "1 vector 1024-d fp16": gb_por_millon(1, 1024, 16), "1 vector 1024-d binario": gb_por_millon(1, 1024, 1)}
    print({k: round(v, 3) for k, v in ESQ.items()}); RES["coste"] = ESQ
except (NotImplementedError, TypeError, NameError, AttributeError, KeyError) as e:
    print("⏳ pendiente (completa TODOs 1–4):", type(e).__name__)

## 4. Rúbrica (autoevaluación)
Umbrales **calibrados con la solución de referencia** (FAST_DEV_RUN=True, corpus sintético, semilla 42) con el margen indicado: el entrenamiento del estudiante tiene varianza entre ejecuciones/hardware, de ahí márgenes de ± 0.05–0.07. Con datos reales solo se exigen chequeos estructurales.

In [ ]:
REF = {'ocr_bm25_0': 0.421, 'ocr_lsi_0': 0.898, 'maxsim_0': 0.761, 'mean_0': 0.761, 'maxsim_03': 0.526, 'ocr_lsi_03': 0.898}

def rubrica(res, acc):
    sint = corpus["source"] == "synthetic" and FAST_DEV_RUN; out = []
    def chk(nombre, f):
        try: out.append((nombre, "✅" if f() else "❌"))
        except Exception: out.append((nombre, "⏳"))
    chk("1. test_perdida", lambda: test_perdida().startswith("✅"))
    chk("1b. precisión de lectura de palabras (σ=0) ≥ 0.70", lambda: acc >= 0.70)
    chk("2-3. test_maxsim_tokens", lambda: test_maxsim_tokens().startswith("✅"))
    chk("5. test_coste", lambda: test_coste().startswith("✅"))
    if sint:
        g = lambda rz, n: res[rz]["ic"][n][0]
        chk(f"4a. OCR + BM25 (σ=0) = {REF['ocr_bm25_0']} ± 0.04", lambda: abs(g(0.0, "OCR + BM25") - REF["ocr_bm25_0"]) <= .04)
        chk(f"4b. OCR + LSI (σ=0) = {REF['ocr_lsi_0']} ± 0.04", lambda: abs(g(0.0, "OCR + LSI") - REF["ocr_lsi_0"]) <= .04)
        chk(f"4c. MaxSim visual (σ=0) = {REF['maxsim_0']} ± 0.07", lambda: abs(g(0.0, "MaxSim visual") - REF["maxsim_0"]) <= .07)
        chk(f"4d. MaxSim visual (σ=0.3) = {REF['maxsim_03']} ± 0.08", lambda: abs(g(0.3, "MaxSim visual") - REF["maxsim_03"]) <= .08)
        chk("4e. MaxSim > OCR+BM25 con IC95 > 0 (σ=0) [la semántica gana al léxico]", lambda: res[0.0]["test"]["ci_low"] > 0)
        chk("4f. el ruido degrada el estudiante visual (σ=0.3 < σ=0) y NO al OCR", lambda: g(0.3, "MaxSim visual") < g(0.0, "MaxSim visual") - 0.1 and abs(g(0.3, "OCR + LSI") - g(0.0, "OCR + LSI")) < 0.03)
        chk("5b. coste ColPali-like ≈ 263.7 GB por 1 M de páginas", lambda: abs(res["coste"]["ColPali-like (1030×128 fp16)"] - 263.68) < 0.1)
    else: print("ℹ️ Datos reales o FAST_DEV_RUN=False: se omiten umbrales numéricos.")
    for n, e in out: print(e, n)
    print(f"\n{sum(e == '✅' for _, e in out)}/{len(out)} criterios cumplidos")
rubrica(RES, acc_lectura)

## 5. Recomendación para la CTO (escribe aquí)
1. **¿OCR + texto, visual o ambos?** (cita IC y el efecto del ruido) …
2. **¿Por qué «OCR + BM25» pierde aquí si el OCR es perfecto?** (pista: ¿qué palabras aparecen en la captura y cuáles en la query?) …
3. **Coste:** ¿cuánto ocuparía el índice multi-vector de 1 M de páginas y qué palancas usarías? …

# ⛔ SPOILER — intenta resolverlo primero
Solución de referencia completa y ejecutable. Si te atascas en un TODO, abre antes las pistas 🪜 de ese paso. Al ejecutar se redefinen las funciones y se recalcula `RES`.

In [ ]:
def perdida_destilacion(z, y, objetivos, tau: float = 0.05):
    logits = F.normalize(z, dim=-1) @ F.normalize(objetivos, dim=-1).T / tau
    return F.cross_entropy(logits, y)
corre(test_perdida)
t0 = time.time(); RED, hist = entrena_estudiante(800 if FAST_DEV_RUN else 2500)
x, y = muestra_celdas(600, np.random.default_rng(99), ruido_max=0.0)
with torch.no_grad(): z = RED(x); acc_lectura = float(((F.normalize(z, dim=-1) @ F.normalize(OBJ, dim=-1).T)[:, 1:].argmax(1) + 1 == y)[y > 0].float().mean())
print(f"entrenado en {time.time()-t0:.0f}s · precisión de lectura (σ=0): {acc_lectura:.3f}")

In [ ]:
def consulta_a_tokens(q: str):
    """Texto de la query -> matriz (n_tokens, 48) con los vectores de término del profesor LSI (sin tildes; descarta OOV). None si vacía."""
    ws = [plano(w) for w in re.findall(r"\w+", q.lower())]; vs = [vec_termino[w] for w in ws if w in vec_termino]
    return np.stack(vs).astype(np.float32) if vs else None

def maxsim_paginas(Q, P):
    return np.einsum("qd,npd->nqp", Q, P).max(axis=2).sum(axis=1)
corre(test_maxsim_tokens)

In [ ]:
def gb_por_millon(n_vec, dim, bits): return n_vec * dim * bits / 8 * 1e6 / 1e9

def informe(ruido):
    per = evalua_sistemas(ruido)
    return {"ic": {n: bootstrap_ci(p["ndcg"]) for n, p in per.items()}, "per": per,
            "test": paired_bootstrap(per["MaxSim visual"]["ndcg"], per["OCR + BM25"]["ndcg"])}
corre(test_coste)
RES = {}
for rz in (0.0, 0.3):
    RES[rz] = informe(rz)
    print(f"\n=== ruido σ={rz} ==="); [print(f"  {n:22s} nDCG@10 = {m:.3f}  IC95 [{lo:.3f}, {hi:.3f}]") for n, (m, lo, hi) in RES[rz]["ic"].items()]
    t = RES[rz]["test"]; print(f"  MaxSim − OCR+BM25: Δ={t['diff']:+.3f} IC95[{t['ci_low']:+.3f},{t['ci_high']:+.3f}] p={t['p_value']:.4f}")
RES["coste"] = {"ColPali-like (1030×128 fp16)": gb_por_millon(1030, 128, 16), "Nuestro juguete (12×48 fp32)": gb_por_millon(12, 48, 32), "1 vector 1024-d fp16": gb_por_millon(1, 1024, 16), "1 vector 1024-d binario": gb_por_millon(1, 1024, 1)}
print({k: round(v, 3) for k, v in RES["coste"].items()}); rubrica(RES, acc_lectura)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8)); sis = list(RES[0.0]["ic"]); cols = [GRIS, NARANJA, VERDE, AZUL, ROJO]
for a, rz in zip(ax, (0.0, 0.3)):
    for i, (n, c) in enumerate(zip(sis, cols)):
        m, lo, hi = RES[rz]["ic"][n]; a.bar(i, m, color=c, yerr=[[m - lo], [hi - m]], capsize=3)
    a.set_xticks(range(len(sis))); a.set_xticklabels(sis, rotation=25, ha="right", fontsize=8); a.set_ylim(0, 1.05); a.set_title(f"nDCG@10 con IC95 % · σ={rz}")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** (1) *OCR + BM25* queda muy por debajo aunque el OCR sea perfecto: **el cuello de botella no es leer, es el vocabulario** (las queries usan sinónimos que no están impresos). (2) *OCR + LSI* (texto semántico) es el mejor sistema, y con σ=0 el **estudiante visual se acerca a su profesor** pero no lo supera: destilar no crea información nueva. (3) Con ruido (σ=0.3) el estudiante visual se degrada mientras que el OCR por plantillas (filtro adaptado) no: un OCR clásico es un baseline duro cuando el texto es legible. (4) El **vector medio** compite bien aquí porque las queries son *temáticas* (bolsa de términos semánticos); en la lección, con queries *conjuntivas* («palabra A y palabra B en la misma página»), la media fallaba: la métrica depende del tipo de consulta.
> Recomendación honesta para Políglota: OCR + embedder de texto como base; **añadir índice visual solo para páginas con contenido no textual** (diagramas/capturas) y medirlo por tipo de consulta.

## 🚀 Retos extra (nivel experto)
1. **Binariza** los patches (signo) y calcula MaxSim con producto en {−1,+1}: ¿cuánto nDCG pierdes y cuánto ahorras (§ coste)?
2. **Token pooling:** agrupa los 12 patches en 6 (media de pares adyacentes) y mide calidad vs almacenamiento.
3. **Pre-filtro + reordenado:** usa el vector medio para quedarte con los 20 mejores y reordena con MaxSim; compara latencia (`time.perf_counter`) y nDCG.
4. **Queries conjuntivas:** crea un slice de queries con 2 términos *clave* de la misma página y re-mide vector medio vs MaxSim con IC (conecta con la lección).
5. **ColPali real (GPU, con red):** renderiza 200 páginas de documentación como PNG (por ejemplo con `matplotlib` o capturas de tu docs), indexa con `colpali-engine` y compara con OCR+BM25 usando `poliglota_eval`; o evalúa en un subconjunto de **ViDoRe** (`vidore/*`) con su paquete de benchmark.
6. **Fusión híbrida:** RRF entre *OCR + LSI* y *MaxSim visual*: ¿mejora con IC pareado y Holm?
7. **Embedder unificado:** si tienes `GEMINI_API_KEY`, embebe 50 capturas reales y compara con tu pipeline (verifica límites/versión del modelo en la documentación vigente).

## 🧠 Reflexión
- ¿Qué tipos de página de la documentación de Políglota justificarían un índice multi-vector y cuáles no?
- Si el almacenamiento es el problema, ¿prefieres binarizar, hacer pooling, o indexar solo páginas con figuras? ¿Cómo lo decidirías con datos?
- ¿Qué cambia en la evaluación (módulo 15) cuando las queries son *visuales* («¿qué diagrama muestra el flujo de auth?») y los qrels dependen de regiones de la página?
- ¿Qué riesgos de privacidad introducen las capturas de pantalla (módulo 18)?

➡️ **Siguiente:** módulo 17 (interpretabilidad y geometría): abrirás la «caja negra» del embedding de Políglota con *probing* y *sparse autoencoders*.